# Build the AI Processor with YOLOv10n

**Author:** Antoine ([@antowan](https://github.com/antowan))

This hands-on tutorial builds **AIPV1EMI0001**, a `car` / `person` / `truck` detector, with YOLOv10n on an
EdgeAI Station: export the model to ONNX, build the parser and the TensorRT engine, validate and
package the AI Processor, then run it in a real pipeline. Each cell is one visible command or
a short check; run them from top to bottom.

**YOLOv10n** comes from Tsinghua University's YOLOv10 research and is distributed through the
[Ultralytics](https://docs.ultralytics.com/models/yolov10/) package. Its key idea for deployment is
an **NMS-free** head: the network is trained to propose one box per object, so no
non-maximum suppression step is needed after inference. That makes the exported graph a plain
network (no plugin node), which is simpler to run and to measure.

The other two flavors follow exactly the same steps: [`01-build-yolov7.ipynb`](01-build-yolov7.ipynb) and [`03-build-yolo26.ipynb`](03-build-yolo26.ipynb).
To understand the concepts first (what an AI Processor is, how the GPU pipeline works, how the
models compare), read [`00-presentation.ipynb`](00-presentation.ipynb) and
[`04-compare-models.ipynb`](04-compare-models.ipynb).

### Learning goals

- Trace how the manifest, the model and its configuration, the callback and the Node-RED project become one AI Processor.
- Export an official model to ONNX, compile it for this GPU and check that it loads.
- Tell apart static checks, an engine smoke test and a real pipeline test.

This is a hands-on build of one existing AI Processor, not a model-training walkthrough. The weights are the
publisher's official COCO weights; the AI Processor only reports three of the 80 COCO classes.

### What this model outputs

The exported graph has a single output:

| Output | Shape (per image) | Meaning |
|---|---|---|
| `output0` | `[300, 6]`, float32 | the 300 best candidates, each `x1, y1, x2, y2, score, class index` in pixels of the 640×640 network input |

There is nothing to suppress: the parser (`NvDsInferParseYoloEndToEnd`) keeps the rows whose class is
car, person or truck and whose score passes the threshold. Ultralytics only exports this head when you
ask for it with `nms=False` (see `flavors/yolov10/flavor.json`); without that argument you get the
raw 84-channel head that needs NMS.

### What you build

| Path in `/data/aip/AIPV1EMI0001` | Role | Section |
|---|---|---|
| `tmp/export/yolov10/` | The ONNX model exported from the official weights (not committed) | 2 |
| `models/AIPV1EMI0001_Yolo_Detector/1/` | The TensorRT FP16 engine compiled on this station | 3 |
| `models/AIPV1EMI0001_Yolo_Detector/libnvds_infercustomparser.so` | C++ parser: model output to DeepStream detections | 3 |
| `manifest.json`, `configs/`, `models/.../config.pbtxt` | Identity, `nvinferserver` and tracker settings; written for you by `tools/flavors.py use` | 1, 5 |
| `callback.py` | Turns per-frame metadata into per-second counts for Node-RED | 8 |
| `AIPV1EMI0001_dist.tgz` | Package built by EAIS Tools, with extra files declared in `.include` | 6 |

### What this notebook proves

The engine loads and runs on this station; the configuration checks pass; EAIS Tools builds and exports
the package; the real pipeline produces valid callback messages **with detections** on a sample video.
It does **not** prove detection accuracy on your scenes, capacity in streams (`eais-tools check perf`) or long-run
stability. For reference, on the 5,000 COCO val2017 images the earlier TensorRT FP16 engine of this flavor measured **38.34** mAP 50:95 over the 80 COCO classes (the YOLOv10 paper reports 38.5 for the same weights) and **40.54** over the three classes of this AI Processor. The separate idle latency rerun measured 5.5 ms per image at batch 1 and 3.8 ms at batch 8 on rebuilt engines that were not rescored for accuracy (see `04-compare-models.ipynb`).

### Steps

```mermaid
flowchart TD
    S0["0. Setup"] --> S1["1. Select the flavor<br/>tools/flavors.py use"]
    S1 --> S2["2. Export<br/>official weights to ONNX"]
    S2 --> S3["3. Build<br/>parser (make) and engine (trtexec)"]
    S3 --> S4["4. Smoke test<br/>the engine loads and runs"]
    S4 --> S5["5. Validate<br/>eais-tools aip check + config validators"]
    S5 --> S6["6. Package<br/>eais-tools aip build / export"]
    S6 --> S7["7. Scan<br/>ms3 finds this AIP"]
    S7 --> S8["8. Pipeline test<br/>decode, infer, track, callback"]
    S8 --> S9["9. Cleanup"]
```

Mermaid diagrams render in JupyterLab 4.1+ and on GitHub.

## 0. Setup

Run this notebook in the station's Jupyter environment (**App-Builder**, `ms8_jupyter`): it provides
`eais-tools`, CUDA, the DeepStream headers and `trtexec`, and shares `/data/aip` (the AI Processors)
with the station services.

The station only discovers AI Processors under `/data/aip/<unique_id>`. On a fresh station, from a Jupyter terminal:

```bash
[ ! -e /data/aip/AIPV1EMI0001 ] || { echo "AIP directory already exists; coordinate the upgrade"; exit 1; }
cd /data/aip && git clone https://github.com/edge-ai/AIPV1EMI0001.git
```

Then open `/data/aip/AIPV1EMI0001/notebooks/02-build-yolov10.ipynb`. The station watches `/data/aip`; do not move,
replace or remove an existing AIP directory as a setup shortcut, and coordinate upgrades with the station administrator.

Tested on an Orin NX 16 GB with JetPack 6.0 (L4T R36.3), TensorRT 8.6.2, DeepStream 7.0 and `eais-tools`.
The other hardware targets in the manifest have not been independently verified.

The `%%bash` cells read the settings below as environment variables and stop at the first failing command
(`set -euo pipefail`).

| Option | Meaning |
|---|---|
| `FLAVOR` | Which model this notebook builds; it selects `flavors/<FLAVOR>/` |
| `WORKSPACE_MIB` | TensorRT builder workspace limit, see section 3 |
| `CLEANUP_EAIS_RESOURCES = True` | Opt in to the station-wide cleanup command in section 9; leave it off unless the station is reserved for this exercise |

In [ ]:
import os
import sys
from pathlib import Path

FLAVOR = "yolov10"
AIP_ID = "AIPV1EMI0001"
WORKSPACE_MIB = 1024
CLEANUP_EAIS_RESOURCES = False

# The notebook lives in notebooks/, but every command runs from the AI Processor root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert Path.cwd() == Path("/data/aip") / AIP_ID, f"Open this notebook from /data/aip/{AIP_ID}/notebooks"

sys.path.insert(0, "tools")
import flavors  # reads flavors/<FLAVOR>/flavor.json, the single source of truth for this model

flavor = flavors.load(Path.cwd(), FLAVOR)
MODEL_DIR = f"models/{AIP_ID}_Yolo_Detector"
os.environ.update(
    AIP_ID=AIP_ID, FLAVOR=FLAVOR, WORKSPACE_MIB=str(WORKSPACE_MIB), MODEL_DIR=MODEL_DIR,
    ENGINE=f"{MODEL_DIR}/1/{flavor['engine']}",
    TRTEXEC=os.environ.get("TRTEXEC_BIN", "/usr/src/tensorrt/bin/trtexec"),
    MS3="http://ms3_videostreaming:8000",
    CLEANUP_EAIS_RESOURCES=str(int(CLEANUP_EAIS_RESOURCES)),
)
print(f"{flavor['display_name']}: ONNX {flavor['onnx']}, engine {flavor['engine']}, WORKSPACE_MIB={WORKSPACE_MIB}")

In [ ]:
%%bash
set -euo pipefail
for tool in eais-tools make jq curl timeout git "$TRTEXEC"; do
    command -v "$tool" > /dev/null || { echo "missing: $tool"; exit 1; }
done
test -d /opt/nvidia/deepstream/deepstream/sources/includes
head -n 1 /etc/nv_tegra_release
/usr/local/cuda/bin/nvcc --version | grep release
banner=$("$TRTEXEC" --help 2>&1 || true)
[[ $banner =~ TensorRT\ v[0-9]+ ]] && echo "${BASH_REMATCH[0]}"
python3 -c "import torch, onnx, onnxruntime; print('torch', torch.__version__, '| onnx', onnx.__version__, '| onnxruntime', onnxruntime.__version__)"
df -h /data/aip | tail -n 1

## 1. Select the flavor

An AI Processor ships **one** detector. This repository keeps the description of every flavor in
`flavors/<id>/` (`flavor.json` plus the Triton and DeepStream configuration files) and
`tools/flavors.py use` copies the chosen one to the places EAIS Tools reads:

| File | What changes |
|---|---|
| `manifest.json`, `store.json` | AI Processor name, description, licence, model and configuration names |
| `configs/config_infer_primary_detector_<id>.txt` | DeepStream `nvinferserver` settings, including the parser function name |
| `models/AIPV1EMI0001_Yolo_Detector/config.pbtxt` | Triton model configuration: input and output tensors |
| `LICENSE.weights` | Licence text of the packaged weights |

**AGPL-3.0.** Ultralytics code and weights are AGPL-3.0 licensed (`LICENSE.weights`). Read the licence, or ask Ultralytics for an enterprise licence, before you ship a package that contains these weights in a commercial product.

`flavors.py check` fails if any of these files no longer matches the selected flavor.

In [ ]:
%%bash
set -euo pipefail
python3 tools/flavors.py use "$FLAVOR"
python3 tools/flavors.py check
python3 tools/flavors.py list

## 2. Export the model to ONNX

[ONNX](https://onnx.ai) is a framework-neutral model format: it is the portable source from which TensorRT compiles
an engine for this GPU. `tools/export_model.py`:

1. downloads the official **COCO weights** (`yolov10n.pt`) and verifies their SHA-256 before using them,
2. exports them to ONNX with the exporter pinned in `flavor.json` (input `images`, batch × 3 × 640 × 640, FP32),
3. checks that the graph inputs and outputs match what the Triton configuration of the flavor expects.

Nothing is trained here. The model detects the 80 COCO classes; the parser reports `car`, `person` and `truck`.

This flavor is exported with the [`ultralytics`](https://pypi.org/project/ultralytics/) package, pinned to one exact
version in `flavor.json`. The next cell installs it with `--no-deps`, so pip keeps the PyTorch, NumPy and
ONNX Runtime builds that the station already provides instead of replacing them.

In [ ]:
%%bash
set -euo pipefail
python3 -m pip install --user --no-deps -r tools/requirements-export.txt
python3 -c "import ultralytics; print('ultralytics', ultralytics.__version__)"

In [ ]:
%%bash
set -euo pipefail
python3 tools/export_model.py "$FLAVOR"

The cell below reads the graph back with the `onnx` package. Every flavor must expose an `images` input; what the
outputs look like is the one thing that differs between the three flavors, and the Triton configuration and the parser must agree with it.

In [ ]:
import onnx

model = onnx.load(str(flavors.onnx_path(Path.cwd(), flavor)), load_external_data=False)


def describe(value):
    dims = [d.dim_param or d.dim_value or "?" for d in value.type.tensor_type.shape.dim]
    return f"{value.name}: {onnx.TensorProto.DataType.Name(value.type.tensor_type.elem_type)} {dims}"


print("inputs: ", [describe(v) for v in model.graph.input])
print("outputs:", [describe(v) for v in model.graph.output])
print("operators:", len(model.graph.node), "| TensorRT plugin nodes:",
      sorted({node.op_type for node in model.graph.node if node.op_type.startswith("EfficientNMS")}) or "none")

## 3. Build the runtime artifacts

`bin/build.sh yolov10` produces the two files the pipeline loads at run time. We run it twice so you can see each part.

### 3.1 Custom parser

`make` compiles `src/nvdsinfer_customparser/` against this station's CUDA and DeepStream headers.
DeepStream cannot know the layout of a model's output tensors, so this small C++ library converts them into
detection objects. It contains one function per output layout: `NvDsInferParseCustomEfficientNMS` (YOLOv7) and
`NvDsInferParseYoloEndToEnd` (YOLOv10 and YOLO26); the DeepStream configuration of the flavor names the one to call.
It also maps the COCO class indices to this AI Processor's own: COCO `car` (2) becomes `car` = 0, COCO `person` (0) becomes `person` = 1 and COCO `truck` (7) becomes `truck` = 2.

`SKIP_ENGINE=1` builds only the parser, which takes a few seconds.

In [ ]:
%%bash
set -euo pipefail
SKIP_ENGINE=1 bin/build.sh "$FLAVOR"
ls -l "$MODEL_DIR/libnvds_infercustomparser.so"

### 3.2 TensorRT engine

`trtexec` compiles the ONNX graph into an engine optimized for **this** GPU and TensorRT version. It can take several minutes; as a reference, the full YOLOv7-tiny build took about 30 minutes on the station used for this tutorial, which was also running another pipeline. The build log goes to `tmp/engines/<flavor>/build.log`, and its tail is shown on failure.

| Flag | Meaning |
|---|---|
| `--fp16` | Allow FP16 kernels (with higher-precision fallbacks): much faster on this GPU, with little accuracy loss (compare them in `04-compare-models.ipynb`) |
| `--minShapes` / `--optShapes` / `--maxShapes` | Dynamic batch profile 1 / 8 / 8: one engine serves 1 to 8 streams and is tuned for 8 (`max_batch_size` in the configs) |
| `--workspace` | Builder scratch memory limit in MiB. The Jetson shares RAM between CPU and GPU: 4096 MiB was OOM-killed with the station stack running, 1024 succeeded |
| `--timingCacheFile` | Reuse kernel timings to speed up later builds (not an engine cache) |

Changing the batch size or the resolution also means changing `config.pbtxt`, the DeepStream configuration and the manifest.
Engines are tied to the TensorRT version and GPU they were built for: do not copy them to a different stack.

In [ ]:
%%bash
set -euo pipefail
bin/build.sh "$FLAVOR"
ls -lh "$ENGINE"

## 4. Smoke test the engine

The quickest proof that the engine file is usable is to load it and run it once. `trtexec --loadEngine` does exactly that.

In [ ]:
%%bash
set -euo pipefail
mkdir -p tmp/build
"$TRTEXEC" --loadEngine="$ENGINE" --shapes=images:1x3x640x640 --duration=1 --warmUp=200 --iterations=1 \
    > tmp/build/smoke-test.log 2>&1 || { tail -n 40 tmp/build/smoke-test.log; exit 1; }
grep -q '&&&& PASSED' tmp/build/smoke-test.log
echo "OK: the engine loads and runs on this station"

## 5. Validate

`eais-tools aip check` verifies the manifest, the required files, the model folders and the callback class.
The next cell runs the EAIS Triton and DeepStream configuration validators directly: the CLI wrappers
(`eais-tools configs triton check` / `configs ds check`) run the same validators but then start a blocking Triton server or a
`gst-launch` test. The NvDCF tracker configuration (OpenCV YAML) gets a basic structure check.

Static checks do not run the tracker on the GPU; the pipeline test in section 8 does. Each layer catches a different class of error:

| Check | Section | Proves | Does not prove |
|---|---|---|---|
| Engine smoke test | 4 | The engine loads and runs on this GPU and TensorRT version | Correct detections |
| Labelled accuracy evaluation | [`04-compare-models.ipynb`](04-compare-models.ipynb) | Detection quality on COCO val2017 | Quality on your own scenes |
| `eais-tools aip check` + config validators | 5 | Manifest, files and the Triton / DeepStream / NvDCF configurations are well formed | That the pipeline runs |
| `eais-tools aip build` / `export` | 6 | EAIS Tools creates the distribution and the installable archive | That the pipeline runs |
| `eais-tools pipeline test` + callback check | 8 | Decode, inference, tracking and callback produce valid messages with detections | Accuracy, capacity or long-run stability |

In [ ]:
%%bash
set -euo pipefail
eais-tools aip check --uuid "$AIP_ID"

In [ ]:
import json
import yaml
from eais_tools.ds_config_validator import validate_ds_config
from eais_tools.services.configs_service import validate_model_config
from eais_tools.triton_config_validator import validate_specified_triton_config

manifest = json.loads(Path("manifest.json").read_text())
detector, tracker = manifest["primary_processor"], manifest["tracker_engine"]
assert detector["config_plan"] == flavor["infer_config"], "run tools/flavors.py use for this flavor"
pbtxt = str(Path(MODEL_DIR, "config.pbtxt").resolve())
triton = validate_specified_triton_config(pbtxt)
assert triton.result and validate_model_config(detector["name"], pbtxt), triton.error_message
ds = validate_ds_config(str(Path("configs", detector["config_plan"]).resolve()))
assert ds.success, ds.message
nvdcf = yaml.safe_load(Path("configs", tracker["config_plan"]).read_text().removeprefix("%YAML:1.0"))
assert {"BaseConfig", "TargetManagement", "StateEstimator", "VisualTracker"} <= nvdcf.keys()
assert Path(tracker["lib_path"]).is_file(), tracker["lib_path"]
print("OK: Triton, DeepStream and NvDCF configurations")

## 6. Package

`eais-tools aip build` copies the runtime files into `dist/`: the manifest, the callback, the root Markdown files,
`resources/logo.png`, `configs/`, and for each model its config, labels, parser and `1/` folder.
**`--include-file .include`** adds files beyond those defaults: licences, documentation, the Node-RED flow and the dashboard.
`.include` lists individual files because EAIS Tools copies directories recursively and does not honor `.gitignore`
(it would ship `node_modules/` or `flows_cred.json`). No additional packaging script is needed.

The package carries the engine of the selected flavor only: `bin/build.sh` removes the engines of other flavors from `models/.../1/`.

In [ ]:
%%bash
set -euo pipefail
eais-tools aip build --uuid "$AIP_ID" --include-file .include

In [ ]:
%%bash
set -euo pipefail
rm -f "${AIP_ID}_dist.tgz"
eais-tools aip export --uuid "$AIP_ID"
sha256sum "${AIP_ID}_dist.tgz"

## 7. Make the AI Processor available to the station

The folder is already in `/data/aip`, so instead of uploading the archive we ask the video-streaming service (ms3) to scan **only** this AI Processor
and check that it found this version. (`AIPV1EMI0001_dist.tgz` is what you install on *other* stations.)

No model-server registration is needed: `nvinferserver` runs the Triton library inside each DeepStream pipeline.
The targeted scan uses `force=false`: it does not rescan unrelated processors or force-unregister this one.
If the scan is refused, ask the station administrator to coordinate any change.

In [ ]:
%%bash
set -euo pipefail
if ! curl -fsS -X POST "$MS3/aiprocessor/scan/$AIP_ID?force=false" > /dev/null; then
    echo "Targeted scan failed. If this AIP is registered with ms7, coordinate with the station administrator." >&2
    exit 1
fi
AIP=$(curl -fsS "$MS3/aiprocessors" | jq -ec --arg id "$AIP_ID" '.[] | select(.unique_id == $id)')
test "$(jq -r .version <<< "$AIP")" = "$(jq -r .version manifest.json)"
echo "OK: ms3 lists $AIP_ID v$(jq -r .version <<< "$AIP") (pk $(jq -r .pk <<< "$AIP"))"

## 8. Run a pipeline test

**`eais-tools pipeline test`** is the runtime acceptance test: it uploads a sample video and runs a station-managed pipeline with this AI Processor's
inference, NvDCF tracker and callback (headless), recording the callback messages to `tmp/callback_dump.json`. It listens until interrupted,
so `timeout` stops it with `SIGINT` after 90 s (exit code 124 is expected). `--retain_resources` keeps the pipeline running so you can look at it afterwards.

The tool replaces any existing `eais_*` test resources. The preflight refuses to run if such resources already exist; do not run this test
concurrently with another `eais_*` test on the station.

In [ ]:
%%bash
set -euo pipefail
eais_resources() { curl -fsS "$MS3/$1" | jq '[.[] | select(.name | startswith("eais_"))] | length'; }
[ "$(eais_resources pipelines)" = 0 ] && [ "$(eais_resources medias)" = 0 ] || {
    echo "eais_* pipelines or media already exist: another test owns them"; exit 1; }
mkdir -p tmp && rm -f tmp/callback_dump.json
status=0
COLUMNS=160 timeout --signal=INT --kill-after=30 90 eais-tools pipeline test --aip "$AIP_ID" \
    --inputs_num 1 --callback_output --retain_resources --dump_file tmp/callback_dump.json || status=$?
[ "$status" = 0 ] || [ "$status" = 124 ]

A started pipeline is not a pass: the next cell requires **detected objects** in callback messages that follow the
[callback contract](../doc/USER_MANUAL.md#callback-output-metadata-contract): `{since, until, data}` records, where `data` maps each source to per-class counts and analytics.

In [ ]:
import json
import requests

text, pos, records, decoder = Path("tmp/callback_dump.json").read_text(), 0, [], json.JSONDecoder()
while pos < len(text):  # the dump is a sequence of JSON arrays, one per message
    if text[pos].isspace():
        pos += 1
        continue
    batch, pos = decoder.raw_decode(text, pos)
    records.extend(batch)
assert records, "no callback messages received"

peak = {c: 0 for c in manifest["primary_processor"]["classes"]}
for record in records:
    assert {"since", "until", "data"} <= record.keys(), record.keys()
    for source in record["data"].values():
        assert source["obj_counter"].keys() == peak.keys(), source["obj_counter"].keys()
        assert {"objInROIcnt", "objLCCurrCnt"} <= source["analytics_obj_cnt"].keys()
        for c in peak:
            peak[c] = max(peak[c], source["obj_counter"][c]["max_val"])
print(f"{len(records)} callback records, peak objects per frame: {peak}")
assert any(peak.values()), "no objects detected"

pipelines = requests.get(f"{os.environ['MS3']}/pipelines", timeout=30).json()
(PIPELINE_ID,) = [p["pk"] for p in pipelines if p["name"] == "eais_pipeline"]
os.environ["PIPELINE_ID"] = PIPELINE_ID
print(f"OK: test pipeline {PIPELINE_ID} delivers detections and valid metadata")

## 9. Cleanup

Station cleanup is **opt-in** (`CLEANUP_EAIS_RESOURCES = True` in section 0). The command `eais-tools pipeline clean` is global: it deletes **all**
`eais_*` pipelines and media, not just the resources created by this notebook. Its guard verifies the pipeline ID but can only limit media to at most one item;
it cannot prove media ownership if another process is active. Enable cleanup only when the station is reserved for this exercise and no other
`eais_*` test is running. When skipped, the test pipeline is retained. Node-RED deployment is a separate workflow in [`logic/README.md`](../logic/README.md).

Kept: `AIPV1EMI0001_dist.tgz` and `dist/` (download the archive to install it on another compatible station), the engine and the parser
(ms3 copies the AIP folder each time a pipeline starts), and the ONNX model in `tmp/export/`.

To try another flavor, open its notebook: it selects the flavor, exports and builds its own engine and replaces the one in `models/.../1/`.

In [ ]:
%%bash
set -euo pipefail
if [ "${CLEANUP_EAIS_RESOURCES:-0}" = 1 ]; then
    curl -fsS "$MS3/pipelines" | jq -e --arg id "$PIPELINE_ID" \
        '[.[] | select(.name | startswith("eais_")) | .pk] == [$id]' > /dev/null
    curl -fsS "$MS3/medias" | jq -e '[.[] | select(.name | startswith("eais_"))] | length <= 1' > /dev/null
    eais-tools pipeline clean
    curl -fsS "$MS3/pipelines" | jq -e --arg id "$PIPELINE_ID" 'all(.[]; .pk != $id)' > /dev/null
else
    echo "Skipped station-wide pipeline cleanup; set CLEANUP_EAIS_RESOURCES=True only on a reserved station."
fi
sha256sum "${AIP_ID}_dist.tgz"